# Theorem 7.4 numerical check (core version of Paper 1)

Simulation only: no data, no downloads, no keys. It checks five things about the identification theorem for position-limited traders:

1. **T1:** populations with the same signed capacity measure give identical responses to every probe.
2. **T2:** populations with the same kernel stay identical until the first trader reaches its limit.
3. **T3:** the constructive recovery returns every memory, leverage and capacity atom.
4. **T4:** the Taylor route for tanh holds.
5. **T5:** a group can be detected before its own limit is reached. This is why part (ii) of the theorem was reworded.

The cell runs it for three seeds, takes about 3 minutes, and downloads `thm74_check_colab.zip`. This checks the theorem numerically. It does not replace a mathematician reading the proof.

In [ ]:
# Theorem 7.4 (core version of Paper 1): numerical check. Simulation only, no market data, nothing to download.
# Runs the checker for three seeds (about 1 minute each) and saves thm74_check_colab.zip.
import json, hashlib, zipfile, subprocess, sys, platform, datetime
SRC = "\"\"\"Numerical check of Theorem 7.4 (core version of Paper 1): stress identifies the signed capacity measure.\n\nSimulation only; no market data. Five checks, each recorded as it comes out:\n  T1  populations with the same signed measure mu~ = sum c_i sgn(l_i) delta_(r_i,|l_i|) give identical responses\n      to every probe (tanh and hard limit), including after splitting traders and adding offsetting pairs;\n  T2  populations with the same kernel but different mu~: identical below the first band edge (hard limit), then\n      separated; under tanh, the first-tap gap scales like A^3;\n  T3  constructive recovery under the hard limit: steer the price with probes, read the date-2 flow map along two\n      directions, locate kinks and slope jumps, and recover every atom (r, |l|, net capacity) of mu~;\n  T4  the Taylor route for tanh: the odd Taylor coefficients of the small-probe flow map are tau_{2n-1} sum d s^{2n-1};\n      checked by comparing the cubic coefficient with its prediction;\n  T5  why part (ii) was reworded: a group can be detected before its own band edge is reached, through the kernel\n      constraint, when another group of the same memory saturates first.\nUsage: python3 thm74_numeric_check.py [seed]   (writes thm74_numeric_check_results.json)\n\"\"\"\nimport json, sys\nimport numpy as np\n\na, kappa = 0.5, 0.8\nSHAPES = {\"tanh\": np.tanh, \"clip\": lambda u: np.clip(u, -1.0, 1.0)}\n\n\ndef respond(pop, v, psi):\n    \"\"\"Price response of the probing experiment: x_t = a x_{t-1} + kappa (sum_i c_i psi(l_i m_{i,t}) + v_t).\"\"\"\n    r = np.array([p[0] for p in pop]); c = np.array([p[1] for p in pop]); l = np.array([p[2] for p in pop])\n    m = np.zeros(len(pop)); x = np.zeros(len(v)); xp = 0.0\n    for t in range(len(v)):\n        x[t] = a * xp + kappa * (np.sum(c * psi(l * m)) + v[t])\n        m = r * m + x[t]; xp = x[t]\n    return x\n\n\ndef flow_map_date2(pop, x0, x1, psi):\n    \"\"\"Steer the price to (x0, x1) with probes, then read the date-2 aggregate flow from the response.\"\"\"\n    r = np.array([p[0] for p in pop]); c = np.array([p[1] for p in pop]); l = np.array([p[2] for p in pop])\n    v0 = x0 / kappa\n    m = r * 0.0 + x0                                    # m_1 = x_0\n    F1 = np.sum(c * psi(l * m))\n    v1 = (x1 - a * x0) / kappa - F1                     # probe that realises x1\n    x1_real = a * x0 + kappa * (F1 + v1)\n    m = r * m + x1_real                                 # m_2 = r x_0 + x_1\n    F2 = np.sum(c * psi(l * m))\n    x2 = a * x1_real + kappa * F2                       # v_2 = 0\n    return (x2 - a * x1_real) / kappa                   # read back F_2 from the response\n\n\ndef mu_tilde(pop, nd=10):\n    out = {}\n    for r, c, l in pop:\n        if l == 0: continue\n        k = (round(r, nd), round(abs(l), nd)); out[k] = out.get(k, 0.0) + c * np.sign(l)\n    return {k: v for k, v in out.items() if abs(v) > 1e-12}\n\n\ndef random_population(rng, n_atoms):\n    rates = rng.choice([0.2, 0.4, 0.6, 0.8], size=n_atoms)\n    lev = rng.choice([0.05, 0.08, 0.12, 0.2, 0.3], size=n_atoms)\n    pop = []\n    for r, lam in zip(rates, lev):\n        c = float(rng.uniform(0.2, 1.5)); sgn = 1.0 if rng.random() < 0.8 else -1.0\n        pop.append((float(r), c, sgn * float(lam)))\n    return pop\n\n\ndef stable(pop):\n    return a + kappa * sum(c * abs(l) / (1 - r) for r, c, l in pop) < 1\n\n\ndef main(seed=20261108):\n    rng = np.random.default_rng(seed); out = {\"seed\": seed}\n    # T1: same mu~, different decompositions\n    gaps = {k: 0.0 for k in SHAPES}; n1 = 0\n    while n1 < 40:\n        P = random_population(rng, 4)\n        if not stable(P): continue\n        Q = []\n        for r, c, l in P:                               # split each trader and add an offsetting pair\n            f = rng.uniform(0.2, 0.8); e = rng.uniform(0.05, 0.3)\n            Q += [(r, f * c, l), (r, (1 - f) * c + e, l), (r, e, -l)]\n        for name, psi in SHAPES.items():\n            for _ in range(5):\n                v = rng.normal(0, 20, 30)\n                gaps[name] = max(gaps[name], float(np.max(np.abs(respond(P, v, psi) - respond(Q, v, psi)))))\n        n1 += 1\n    out[\"T1_same_mu_tilde_max_gap\"] = gaps\n    out[\"T1_pass\"] = all(g < 1e-9 for g in gaps.values())\n    # T2: same kernel, different mu~ (two groups at memory 0.7)\n    P = [(0.7, 1.0, 0.1)]; Pp = [(0.7, 0.25, 0.2), (0.7, 1.0, 0.05)]\n    rows = []\n    for A in [0.5, 1, 2, 4, 5, 6, 8, 12]:\n        v = np.zeros(30); v[0] = A\n        rows.append({\"A\": A, \"gap_clip\": float(np.max(np.abs(respond(P, v, SHAPES[\"clip\"]) - respond(Pp, v, SHAPES[\"clip\"])))),\n                     \"gap_tanh_tap1\": float(abs(respond(P, v, np.tanh)[1] - respond(Pp, v, np.tanh)[1]))})\n    out[\"T2_rows\"] = rows\n    small = [rw for rw in rows if rw[\"A\"] <= 1]\n    out[\"T2_tanh_tap1_slope_small_A\"] = float(np.polyfit(np.log([rw[\"A\"] for rw in small]), np.log([rw[\"gap_tanh_tap1\"] for rw in small]), 1)[0])\n    first = next(rw[\"A\"] for rw in rows if rw[\"gap_clip\"] > 0)\n    out[\"T2_clip_zero_below\"] = first\n    out[\"T2_pass\"] = all(rw[\"gap_clip\"] == 0 for rw in rows if rw[\"A\"] < first) and first > 4\n    # T3: constructive recovery under the hard limit, random populations, two directions\n    results = []\n    n3 = 0\n    while n3 < 25:\n        P = random_population(rng, int(rng.integers(2, 5)))\n        if not stable(P): continue\n        truth = mu_tilde(P)\n        rec = {}\n        for th in (0.3, 0.9):\n            us = np.linspace(0, 120, 120001)\n            g = np.array([flow_map_date2(P, th * u, u, SHAPES[\"clip\"]) for u in us])\n            sl = np.diff(g) / np.diff(us); jumps = np.where(np.abs(np.diff(sl)) > 1e-7)[0]\n            kinks = []\n            for j in jumps:\n                uk = us[j + 1]\n                if kinks and uk - kinks[-1][0] < 0.01: continue\n                d = 0.002\n                sL = (flow_map_date2(P, th * (uk - d), uk - d, SHAPES[\"clip\"]) - flow_map_date2(P, th * (uk - 2 * d), uk - 2 * d, SHAPES[\"clip\"])) / d\n                sR = (flow_map_date2(P, th * (uk + 2 * d), uk + 2 * d, SHAPES[\"clip\"]) - flow_map_date2(P, th * (uk + d), uk + d, SHAPES[\"clip\"])) / d\n                kinks.append((uk, sR - sL))\n            rec[th] = [(1 / uk, -jump * uk) for uk, jump in kinks]          # (s_k, d_k)\n        # match the two directions by net capacity d (direction-free), solve for (r, lambda)\n        got = {}\n        for s1, d1 in rec[0.3]:\n            cand = min(rec[0.9], key=lambda sd: abs(sd[1] - d1)) if rec[0.9] else None\n            if cand is None: continue\n            s2 = cand[0]; q = s1 / s2\n            r = (q - 1) / (0.9 * 0 + 0.3 - q * 0.9) if abs(0.3 - q * 0.9) > 1e-12 else np.nan\n            lam = s1 / (r * 0.3 + 1)\n            got[(round(float(r), 3), round(float(lam), 3))] = round(float(d1), 3)\n        tr = {(round(k[0], 3), round(k[1], 3)): round(v, 3) for k, v in truth.items()}\n        # match atoms with a tolerance set by the kink grid (step 0.001 in u)\n        def near(k1, k2): return abs(k1[0] - k2[0]) <= 5e-3 and abs(k1[1] - k2[1]) <= 5e-3\n        ok = len(tr) == len(got) and all(any(near(k, g) and abs(tr[k] - got[g]) < 5e-3 for g in got) for k in tr)\n        results.append({\"truth\": {str(k): v for k, v in tr.items()}, \"recovered\": {str(k): v for k, v in got.items()}, \"match\": ok})\n        n3 += 1\n    out[\"T3_recovery\"] = results\n    out[\"T3_matches\"] = sum(r_[\"match\"] for r_ in results); out[\"T3_cases\"] = len(results)\n    # T4: Taylor route for tanh: cubic coefficient of the date-2 flow map along a direction\n    P = [(0.7, 1.0, 0.1)]; Pp = [(0.7, 0.25, 0.2), (0.7, 1.0, 0.05)]; th = 0.5\n    us = np.linspace(-0.3, 0.3, 61)\n    gdiff = np.array([flow_map_date2(P, th * u, u, np.tanh) - flow_map_date2(Pp, th * u, u, np.tanh) for u in us])\n    coef = np.polyfit(us, gdiff, 7)                     # highest power first\n    def mu_list(pop): return [(c * np.sign(l), abs(l) * (r * th + 1)) for r, c, l in pop]\n    pred3 = -1 / 3 * (sum(d * s ** 3 for d, s in mu_list(P)) - sum(d * s ** 3 for d, s in mu_list(Pp)))\n    out[\"T4_cubic_coef_fit\"] = float(coef[-4]); out[\"T4_cubic_coef_pred\"] = float(pred3)\n    out[\"T4_linear_coef_fit\"] = float(coef[-2])\n    out[\"T4_pass\"] = abs(coef[-4] - pred3) < 1e-6 * max(1, abs(pred3)) + 1e-9 and abs(coef[-2]) < 1e-9\n    # T5: indirect detection before a group's own band edge\n    P = [(0.7, 1.0, 0.1), (0.7, 1.0, 0.4)]              # kernel weight at 0.7: 0.1 + 0.4 = 0.5 ... (stability below)\n    Pp = [(0.7, 2.0, 0.1), (0.7, 0.75, 0.4)]            # 0.2 + 0.3 = 0.5: same kernel, differs at both atoms\n    th = 0.5; s_hi, s_lo = 0.4 * 1.35, 0.1 * 1.35\n    u_mid = 0.5 * (1 / s_hi + 1 / s_lo)                 # beyond the high-leverage edge, before the low-leverage edge\n    out[\"T5_u_edges\"] = [1 / s_hi, 1 / s_lo]\n    out[\"T5_gap_inside_all_bands\"] = float(abs(flow_map_date2(P, th * 1.0, 1.0, SHAPES[\"clip\"]) - flow_map_date2(Pp, th * 1.0, 1.0, SHAPES[\"clip\"])))\n    out[\"T5_gap_between_edges\"] = float(abs(flow_map_date2(P, th * u_mid, u_mid, SHAPES[\"clip\"]) - flow_map_date2(Pp, th * u_mid, u_mid, SHAPES[\"clip\"])))\n    out[\"T5_note\"] = (\"The populations differ at the low-leverage atom too, but the difference is already visible once only the \"\n                      \"high-leverage group has saturated: with the kernel fixed, a change at one atom forces a change elsewhere.\")\n    out[\"ALL_PASS\"] = bool(out[\"T1_pass\"] and out[\"T2_pass\"] and out[\"T3_matches\"] == out[\"T3_cases\"] and out[\"T4_pass\"]\n                           and out[\"T5_gap_inside_all_bands\"] == 0 and out[\"T5_gap_between_edges\"] > 0)\n    return out\n\n\nif __name__ == \"__main__\":\n    seed = int(sys.argv[1]) if len(sys.argv) > 1 else 20261108\n    res = main(seed)\n    json.dump(res, open(\"thm74_numeric_check_results.json\", \"w\"), indent=1, default=str)\n    print(json.dumps({k: v for k, v in res.items() if k not in (\"T3_recovery\", \"T2_rows\")}, indent=1, default=str))\n"
SHA = "709f8c577312283f771a1cd139950ab9b30e7f0c1b83c3800b11cb13bb1ce778"
assert hashlib.sha256(SRC.encode()).hexdigest() == SHA, "checker source changed"
open("thm74_numeric_check.py", "w").write(SRC)
summary = {"checker_sha256": SHA, "python": sys.version, "platform": platform.platform(),
           "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "runs": {}}
for seed in (20261108, 1, 2):
    subprocess.run([sys.executable, "thm74_numeric_check.py", str(seed)], check=True)
    res = json.load(open("thm74_numeric_check_results.json"))
    json.dump(res, open(f"thm74_results_seed{seed}.json", "w"), indent=1, default=str)
    summary["runs"][str(seed)] = {k: res[k] for k in ("T1_pass", "T2_pass", "T2_clip_zero_below", "T2_tanh_tap1_slope_small_A",
                                   "T3_matches", "T3_cases", "T4_pass", "T5_gap_inside_all_bands", "T5_gap_between_edges", "ALL_PASS")}
    print("seed", seed, json.dumps(summary["runs"][str(seed)]))
json.dump(summary, open("thm74_check_summary.json", "w"), indent=1, default=str)
with zipfile.ZipFile("thm74_check_colab.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["thm74_check_summary.json", "thm74_numeric_check.py"] + [f"thm74_results_seed{s}.json" for s in (20261108, 1, 2)]:
        z.write(f)
print("ALL SEEDS PASS:", all(r["ALL_PASS"] for r in summary["runs"].values()))
try:
    from google.colab import files; files.download("thm74_check_colab.zip")
except Exception:
    print("saved thm74_check_colab.zip")
